# 01 · Ingest and chunk

Loads the 18 cleaned .docx files (from `RAG_DATA_DIR`), chunks them and checks the result against the golden sets. The logic lives in `src/rag/`; this notebook only calls it and shows the output. Design notes: [docs/chunking.md](../docs/chunking.md).

In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.rag import config
from src.rag.ingest import load_manifest, load_corpus
from src.rag.chunk import chunk_corpus
from src.rag.report import chunk_stats, golden_coverage

print(config.DATA_DIR)

## Ingest

In [ ]:
manifest = load_manifest(config.MANIFEST)
docs = load_corpus(config.DATA_DIR, config.MANIFEST)
for d in docs:
    n_tab = sum(type(b).__name__ == 'Table' for b in d.blocks)
    print(f'{d.doc_id:24s} blocks={len(d.blocks):4d} tables={n_tab}')

## Chunk

In [ ]:
chunks = chunk_corpus(docs, manifest, max_words=config.MAX_WORDS, min_words=config.MIN_WORDS)
s = chunk_stats(chunks)
{k: v for k, v in s.items() if k != 'per_doc'}

In [ ]:
s['per_doc']

### Size distribution

In [ ]:
import collections
bins = collections.Counter(min(c.n_words // 50 * 50, 350) for c in chunks)
for b in sorted(bins):
    print(f'{b:3d}-{b+49:3d} {"#" * bins[b]}')

### Sample chunks

An article with a table (Điều 24), a table row expanded line by line (Phụ lục vi phạm, row 28), and a guide section split in parts.

In [ ]:
def show(c, n=1200):
    print(c.id, '|', c.loc, '|', c.n_words, 'words')
    print(c.context)
    print('-' * 80)
    print(c.text[:n])
    print('=' * 80)

for c in chunks:
    if c.doc_id == 'qc-dao-tao' and c.article == 24:
        show(c)

In [ ]:
show(next(c for c in chunks if c.doc_id == 'phu-luc-vi-pham' and '\n28. ' in '\n' + c.text))

In [ ]:
for c in chunks:
    if c.doc_id == 'dk-thu-khen':
        show(c, 400)

## Golden-set coverage

Every cited location must map to at least one chunk; otherwise Recall@k could not be computed for that question.

In [ ]:
rows = [json.loads(line) for line in open(config.EVAL_DIR / 'golden.jsonl', encoding='utf-8')]
cov = golden_coverage(rows, chunks, manifest)
answerable = [r for r in rows if r['expected_behavior'] == 'answer']
mapped = {m['id'] for m in cov['mapping']}
print(len(rows), 'questions,', len(answerable), 'answerable,', sum(r['id'] in mapped for r in answerable), 'mapped to chunks')
print('unresolved:', cov['unresolved'])

Write the outputs (same as `python scripts/build_chunks.py`):

In [ ]:
import subprocess
print(subprocess.run([sys.executable, '../scripts/build_chunks.py'], capture_output=True, text=True).stdout[-400:])